# Stage 6: QLoRA Fine-Tuning (Google Colab, T4 GPU)

**Project:** Aspect-Level Customer Review Intelligence via Fine-Tuned Small Language Models

Fine-tunes `Qwen2.5-1.5B-Instruct` with QLoRA on the SemEval Laptops
training pairs prepared in `01_data_prep.ipynb`, per
`Customer_Review_Intelligence_Review0.md` §13. **Designed to run on Colab's
free T4 GPU — Runtime → Change runtime type → T4 GPU before running any cell
below.** GPU, not TPU: QLoRA's 4-bit quantization (`bitsandbytes`) and
Unsloth's fast kernels are CUDA-specific and have no meaningful TPU path.

**Data comes from a direct local upload, not Google Drive.** When the
"Upload input data" cell runs, a browser file picker opens — select all four
files at once from the local repo's `data/processed/`:
`train.jsonl`, `val.jsonl`, `test.jsonl`, `phaseb_extraction_check.jsonl`.
They land in the Colab VM's own disk (`/content/asmta/...`), not Drive.

**Tradeoff, stated plainly:** without Drive, checkpoints saved during the
full training run live only on this ephemeral VM — if Colab disconnects
mid-run (§17 Risk Register calls this "High likelihood"), they're lost and
the run needs restarting. What's still protected against a milder problem
(picking an overfit final epoch) is `load_best_model_at_end` below, which
makes the Trainer automatically keep the best-validation-loss checkpoint in
memory regardless of which epoch finished last — but that only helps if the
session survives to the end of training. If disconnects turn out to be a
real problem, mounting Drive for checkpointing is the fix (§13.3) — left out
here per instruction to keep this local-upload-only.

**Another caveat:** this notebook was authored without a GPU available to
test against, so unlike every other notebook in this project it has not been
executed end to end before being handed to you. `unsloth`/`trl` move
quickly — if a cell errors on an argument name, check that library's current
docs for the installed version; the *shape* of the pipeline (upload → toy
run → full run → save adapter → generate on held-out data → download) is
what matters and is stable even if a keyword argument or two needs
adjusting.

**Run order:** Setup → Upload input data → Load model + LoRA → **Toy run
(10 steps) — confirm this completes before continuing** → Full fine-tune →
Save adapter → Generate on test + phaseb sets → Download results.


## 1. Setup

In [ ]:
%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes

In [ ]:
import json
import random
import shutil
import time
from pathlib import Path

import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from unsloth import FastLanguageModel

SEED = 42  # same seed used throughout this project (see data/processed/MANIFEST.json)
random.seed(SEED)
torch.manual_seed(SEED)

MAX_SEQ_LENGTH = 512  # §13.1; notebook 00 measured a ~150-token worst case, comfortable margin

# The response marker the prompt template ends every input with — used below
# to mask the prompt out of the training loss (completion-only loss, §10.4).
# Every train/val/test record already carries a "prompt" field built from
# the identical template in src/model_config.py — reused verbatim here
# rather than redefined, so there is no risk of the two drifting apart.
RESPONSE_MARKER = "### Response:\n"

# Everything lives under here, in the Colab VM's own (ephemeral) disk — no
# Drive mount. Recreated fresh each session.
WORKDIR = Path("/content/asmta")
DATA_DIR = WORKDIR / "data" / "processed"
CKPT_DIR = WORKDIR / "adapters" / "checkpoints"
FINAL_ADAPTER_DIR = WORKDIR / "adapters" / "final"
OUT_DIR = WORKDIR / "outputs" / "raw_generations" / "finetuned"
for d in [DATA_DIR, CKPT_DIR, FINAL_ADAPTER_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Working directory tree ready at", WORKDIR)


## 2. Upload input data (local)

Select all four files at once in the picker:
`train.jsonl`, `val.jsonl`, `test.jsonl`, `phaseb_extraction_check.jsonl`
(from the local repo's `data/processed/`).


In [ ]:
from google.colab import files as colab_files

print("Select train.jsonl, val.jsonl, test.jsonl, phaseb_extraction_check.jsonl (all four at once):")
uploaded = colab_files.upload()

expected = {"train.jsonl", "val.jsonl", "test.jsonl", "phaseb_extraction_check.jsonl"}
missing = expected - set(uploaded.keys())
if missing:
    raise RuntimeError(f"Missing expected file(s): {missing}. Re-run this cell and select all four.")

# files.upload() writes into the current working directory; move into the
# project's own data layout so the rest of the notebook has stable paths.
for fname in uploaded:
    shutil.move(fname, DATA_DIR / fname)

print("Uploaded and placed in", DATA_DIR)


In [ ]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

train_records = load_jsonl(DATA_DIR / "train.jsonl")
val_records = load_jsonl(DATA_DIR / "val.jsonl")
test_records = load_jsonl(DATA_DIR / "test.jsonl")
phaseb_records = load_jsonl(DATA_DIR / "phaseb_extraction_check.jsonl")

print(f"train={len(train_records)} val={len(val_records)} "
      f"test={len(test_records)} phaseb={len(phaseb_records)}")
assert len(train_records) == 2402 and len(val_records) == 300 and len(test_records) == 300, (
    "Record counts don't match 01_data_prep.ipynb's output — wrong files uploaded?"
)


In [ ]:
# Each train/val record already has "prompt" and "completion" (built by
# 01_data_prep.ipynb) — concatenating them is the full training text.
def to_sft_text(r):
    return r["prompt"] + r["completion"]

train_ds = Dataset.from_dict({"text": [to_sft_text(r) for r in train_records]})
val_ds = Dataset.from_dict({"text": [to_sft_text(r) for r in val_records]})

print(train_ds[0]["text"])


## 3. Load base model (4-bit) and attach LoRA adapters (§13.1)

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,        # auto-selects fp16 on T4 (T4 has no bf16 support, §13.1)
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],  # attention + MLP, §13.1
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,} ({trainable/total:.2%}) — "
      f"expect roughly 0.5-1.5% per §13.1's efficiency table")


In [ ]:
collator = DataCollatorForCompletionOnlyLM(response_template=RESPONSE_MARKER, tokenizer=tokenizer)


## 4. Toy run — 100 examples, 10 steps (§13.3)

*"Do the first run with a 100-example subset and 10 training steps to
verify the whole pipeline executes, then run it properly."* — the single
highest-value habit in the reference doc. **Confirm this cell completes
without error before running the full fine-tune below.**


In [ ]:
toy_train_ds = train_ds.shuffle(seed=SEED).select(range(100))

toy_args = TrainingArguments(
    output_dir="/content/toy_run",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    max_steps=10,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=1,
    seed=SEED,
    report_to="none",
)

toy_trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=toy_train_ds,
    data_collator=collator,
    args=toy_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
)

toy_trainer.train()
print("\nToy run complete — pipeline executes end to end. Proceed to the full run below.")


## 5. Full fine-tune (§13.1–§13.2)

3 epochs, effective batch size 8 (2 × 4 grad-accum), checkpointed every
epoch to this VM's local disk. `load_best_model_at_end=True` makes the
Trainer automatically restore the lowest-validation-loss checkpoint into
`model` after training finishes — so "save the final adapter" below saves
the best epoch, not necessarily the last one, directly implementing §13.2's
"stop and use the earlier checkpoint" without relying on a manual read of
the log.


In [ ]:
full_args = TrainingArguments(
    output_dir=str(CKPT_DIR),
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    seed=SEED,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=collator,
    args=full_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
)

train_result = trainer.train()


In [ ]:
print("Validation loss by epoch:")
for entry in trainer.state.log_history:
    if "eval_loss" in entry:
        print(f"  epoch {entry.get('epoch'):.1f}: eval_loss={entry['eval_loss']:.4f}")
print(f"\nBest checkpoint (now loaded into `model`): {trainer.state.best_model_checkpoint}")


## 6. Save the final adapter

In [ ]:
model.save_pretrained(str(FINAL_ADAPTER_DIR))
tokenizer.save_pretrained(str(FINAL_ADAPTER_DIR))
print(f"Saved adapter (best-validation-loss checkpoint) to {FINAL_ADAPTER_DIR}")


## 7. Generate on the held-out test set + the phaseB extraction-check set

Same greedy decoding (`do_sample=False`), same generation cap
(`max_new_tokens=200`), same two files the zero-shot baseline notebook
scored — so this drops directly into the existing `src/eval_metrics.py`
scoring code once brought back to the local repo.


In [ ]:
FastLanguageModel.for_inference(model)  # Unsloth's faster inference path

def generate_one(prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    return tokenizer.decode(out[0][input_len:], skip_special_tokens=True)

def run_inference(records, label):
    results = []
    t0 = time.time()
    for i, r in enumerate(records):
        raw = generate_one(r["prompt"])
        results.append({"sentence_id": r["sentence_id"], "text": r["text"], "raw_output": raw})
        if (i + 1) % 50 == 0 or (i + 1) == len(records):
            print(f"  [{label}] {i+1}/{len(records)} — {time.time()-t0:.0f}s elapsed")
    return results

test_generations = run_inference(test_records, "test")
phaseb_generations = run_inference(phaseb_records, "phaseb")


## 8. Save outputs and download everything back

In [ ]:
def write_jsonl(path, records):
    with open(path, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")

write_jsonl(OUT_DIR / "test_generations.jsonl", test_generations)
write_jsonl(OUT_DIR / "phaseb_generations.jsonl", phaseb_generations)
print(f"Saved to {OUT_DIR}")


In [ ]:
# Zip only what's needed back in the local repo — the final adapter and the
# two generation files. Deliberately excludes adapters/checkpoints/ (the two
# intermediate epoch saves): they were only useful mid-run for
# load_best_model_at_end, not needed once the best one is already in
# FINAL_ADAPTER_DIR, and skipping them keeps the download small and fast.
zip_base = "/content/asmta_finetune_outputs"
shutil.make_archive(zip_base, "zip", root_dir=WORKDIR, base_dir="adapters/final")
# Append the outputs folder into the same archive.
shutil.make_archive("/content/asmta_finetune_outputs_outputs", "zip", root_dir=WORKDIR, base_dir="outputs")

colab_files.download(zip_base + ".zip")
colab_files.download("/content/asmta_finetune_outputs_outputs.zip")
print("Two downloads triggered: the adapter, and the raw generations.")


### Bring these back to the local repo

Unzip the two downloaded files and merge:
- the adapter zip's contents → local `adapters/final/`
- the outputs zip's `outputs/raw_generations/finetuned/{test_generations,phaseb_generations}.jsonl` → local `outputs/raw_generations/finetuned/`

Then, back in the local repo (no GPU needed from here on): `notebooks/03_evaluate.ipynb`
reuses `src/eval_metrics.py` — the same code that scored the zero-shot
baseline — against these fine-tuned generations, on the identical 300-sentence
test set, to produce the actual zero-shot-vs-fine-tuned comparison.
